# Benford's Law on the Stellar DEX — An Interactive Explainer

**LedgerLens** uses Benford's Law as one of its primary signals for detecting
wash trading on the Stellar decentralised exchange.  This notebook walks through
the technique step by step, demonstrates it on realistic synthetic Stellar trade
data, and shows how the signal degrades when a wash-trade ring starts operating.

> **No real wallet addresses are used.**  All data come from the synthetic
> dataset generated by `scripts/generate_synthetic_dataset.py`.

---

### What you will learn

1. What Benford's Law is and why it applies to organic trading activity
2. How to compute the three LedgerLens Benford metrics (chi-square, Z-scores, MAD)
3. How clean-wallet distributions differ visually from wash-trader distributions
4. How the MAD score evolves over time as a wash-trade ring builds up
5. How the rolling-window analysis (1 h → 30 d) behaves in practice


In [ ]:
# Standard library
import sys
import math
import pathlib
import importlib.util

# Third-party
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch

# ── Load benford_engine directly (avoids heavy detection/__init__ chain) ──────
ROOT = pathlib.Path("..").resolve()

def _load_module(name, rel_path):
    path = ROOT / rel_path
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

# Stub out 'config' so benford_engine's lazy import doesn't fail in CI
import types
_cfg_stub = types.SimpleNamespace(
    MIN_TRADES_FOR_SCORING=5,
    BENFORD_WINDOWS_HOURS=[1, 4, 24, 168, 720],
    BENFORD_CI_ENABLED=False,
    GNN_EMBEDDING_DIM=32,
    ASSET_BENFORD_WINDOWS={},
)
_cfg_module = types.ModuleType("config")
_cfg_module.config = _cfg_stub
sys.modules.setdefault("config", _cfg_module)

be = _load_module("benford_engine", "detection/benford_engine.py")

BENFORD_EXPECTED          = be.BENFORD_EXPECTED
MAD_NONCONFORMITY_THRESHOLD = be.MAD_NONCONFORMITY_THRESHOLD
leading_digits            = be.leading_digits
observed_distribution     = be.observed_distribution
chi_square_statistic      = be.chi_square_statistic
z_scores                  = be.z_scores
mad_score                 = be.mad_score
compute_benford_metrics   = be.compute_benford_metrics

# ── Colourblind-accessible palette (Okabe-Ito, 2008) ──────────────────────────
CB = {
    "blue":   "#0072B2",
    "orange": "#E69F00",
    "green":  "#009E73",
    "red":    "#D55E00",
    "purple": "#CC79A7",
    "sky":    "#56B4E9",
    "yellow": "#F0E442",
    "black":  "#000000",
}

matplotlib.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# Use non-interactive backend for CI
plt.switch_backend("Agg")

print("Imports OK.  matplotlib:", matplotlib.__version__)
print("pandas:", pd.__version__, "  numpy:", np.__version__)
print(f"MAD non-conformity threshold: {MAD_NONCONFORMITY_THRESHOLD}")


---
## 1  What is Benford's Law?

In 1881 the astronomer Simon Newcomb noticed that the early pages of logarithm
tables were more worn than the later ones — meaning people looked up numbers
beginning with **1** far more often than numbers beginning with **9**.
Frank Benford formalised this in 1938 across dozens of natural datasets.

The expected frequency of leading digit $d$ (where $d \in \{1,\ldots,9\}$) is:

$$P(d) = \log_{10}\!\left(1 + \frac{1}{d}\right)$$

| Digit | Expected frequency |
|------:|-------------------:|
| 1 | 30.1 % |
| 2 | 17.6 % |
| 3 | 12.5 % |
| 4 | 9.7 % |
| 5 | 7.9 % |
| 6 | 6.7 % |
| 7 % | 5.8 % |
| 8 | 5.1 % |
| 9 | 4.6 % |

**Why does organic trading conform?**  Real traders place orders driven by market
prices, which span many orders of magnitude and are influenced by independent
economic factors.  The resulting amount distribution is approximately log-uniform,
which naturally produces the Benford pattern.

**Why do wash traders break it?**  Wash-trading bots typically use fixed lot
sizes, round numbers, or algorithmically generated amounts with limited
variety.  The resulting distribution clusters around a few digit values and
deviates systematically from Benford's Law.


In [ ]:
digits = list(range(1, 10))
expected_freqs = [BENFORD_EXPECTED[d] * 100 for d in digits]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(digits, expected_freqs, color=CB["blue"], alpha=0.85, width=0.6)
ax.set_xlabel("Leading digit")
ax.set_ylabel("Expected frequency (%)")
ax.set_title("Benford's Law — expected leading-digit distribution")
ax.set_xticks(digits)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=1))
for bar, freq in zip(bars, expected_freqs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f"{freq:.1f}%", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()


---
## 2  Loading the synthetic dataset

LedgerLens ships a synthetic labelled feature matrix at
`data/synthetic_dataset.parquet`.  It contains 500 wallet rows — half
legitimate (label 0) and half wash-trading (label 1) — with the same feature
columns as the live pipeline.

Because the parquet stores pre-computed *features* (not raw trade amounts),
this section also **reconstructs plausible raw trade amounts** for each group
so we can demonstrate the digit-distribution plots.  The amounts are generated
using the same statistical assumptions as the simulator: legitimate wallets
draw amounts from a log-uniform distribution (Benford-conforming by
construction), while wash traders draw from distributions clustered around
round numbers or fixed lot sizes.


In [ ]:
DATA_PATH = ROOT / "data" / "synthetic_dataset.parquet"
df = pd.read_parquet(DATA_PATH)

print(f"Dataset shape : {df.shape}")
print(f"Label counts  : {df['label'].value_counts().to_dict()}")
print()
print("Columns (first 15):", df.columns.tolist()[:15])


In [ ]:
# ── Reconstruct illustrative trade-amount series ──────────────────────────────
# Legitimate wallets: log-uniform amounts → Benford-conforming
# Wash traders: fixed lot sizes + small noise → non-conforming

RNG = np.random.default_rng(42)
N_TRADES = 500  # trades per wallet group (pooled)

# Legitimate: log10(amount) ~ Uniform(0, 5)  →  amount in [1, 100 000]
legit_amounts = pd.Series(10 ** RNG.uniform(0, 5, size=N_TRADES))

# Wash trader: 60 % round lots (100, 500, 1 000, 5 000) + 40 % small jitter
lot_sizes = RNG.choice([100.0, 500.0, 1000.0, 5000.0], size=int(N_TRADES * 0.6))
jitter = 10 ** RNG.uniform(0, 3, size=int(N_TRADES * 0.4))
wash_amounts = pd.Series(np.concatenate([lot_sizes, jitter]))

print("Legitimate — sample stats:")
print(legit_amounts.describe().round(2))
print()
print("Wash trader — sample stats:")
print(wash_amounts.describe().round(2))


---
## 3  Visualising the leading-digit distributions

The two groups should look strikingly different.  Legitimate trading closely
tracks the Benford curve; wash trading shows an unnatural spike at digit **1**
(from the 1 000 and 100 lot sizes) and digit **5** (from 500).


In [ ]:
def digit_bar(ax, amounts, label, color, title):
    obs = observed_distribution(amounts)
    obs_pct  = [obs[d] * 100 for d in digits]
    exp_pct  = [BENFORD_EXPECTED[d] * 100 for d in digits]

    x = np.arange(len(digits))
    w = 0.35
    ax.bar(x - w/2, obs_pct,  width=w, label=label,      color=color, alpha=0.85)
    ax.bar(x + w/2, exp_pct,  width=w, label="Benford expected",
           color=CB["black"], alpha=0.35)
    ax.set_xticks(x)
    ax.set_xticklabels(digits)
    ax.set_xlabel("Leading digit")
    ax.set_ylabel("Frequency (%)")
    ax.set_title(title)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
    ax.legend(fontsize=9)

fig, (ax_l, ax_w) = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
digit_bar(ax_l, legit_amounts, "Legitimate wallets", CB["blue"],
          "Legitimate wallets vs. Benford")
digit_bar(ax_w, wash_amounts,  "Wash traders",       CB["red"],
          "Wash traders vs. Benford")
plt.suptitle("Leading-digit distributions — Stellar DEX synthetic data", fontsize=13)
plt.tight_layout()
plt.show()


---
## 4  Computing the three LedgerLens Benford metrics

LedgerLens uses three complementary statistics, each catching a different aspect
of distributional divergence.

### Chi-square statistic
Tests whether the overall digit distribution differs significantly from the
expected one.  Larger values → more deviation.  Under the null hypothesis of
perfect Benford conformance with large *n*, this follows a χ² distribution with
8 degrees of freedom (critical value ≈ 15.5 at p = 0.05).

### Per-digit Z-scores
A per-digit two-sided Z-test (with continuity correction, Nigrini 2012).
Useful for identifying *which* digit is anomalous — a wash trader using
10 000-XLM lots would show a spike on digit 1.

### Mean Absolute Deviation (MAD)
A composite measure: the average absolute difference between the observed and
expected frequency across all nine digits.

| MAD range | Interpretation (Nigrini 2012) |
|-----------|-------------------------------|
| 0.000 – 0.006 | Close conformity |
| 0.006 – 0.012 | Acceptable conformity |
| 0.012 – 0.015 | Marginal conformity |
| > **0.015** | **Non-conformity — flag** |

The `MAD_NONCONFORMITY_THRESHOLD` in LedgerLens is **0.015**.


In [ ]:
chi_legit = chi_square_statistic(legit_amounts)
chi_wash  = chi_square_statistic(wash_amounts)

mad_legit = mad_score(legit_amounts)
mad_wash  = mad_score(wash_amounts)

zs_legit  = z_scores(legit_amounts)
zs_wash   = z_scores(wash_amounts)

print(f"{'Metric':<30} {'Legitimate':>15} {'Wash traders':>15}")
print("-" * 62)
print(f"{'Chi-square statistic':<30} {chi_legit:>15.2f} {chi_wash:>15.2f}")
print(f"{'MAD score':<30} {mad_legit:>15.4f} {mad_wash:>15.4f}")
print(f"{'MAD non-conforming?':<30} {str(mad_legit > MAD_NONCONFORMITY_THRESHOLD):>15}"
      f" {str(mad_wash > MAD_NONCONFORMITY_THRESHOLD):>15}")
print()
print("Per-digit Z-scores:")
print(f"  {'Digit':<8}", end="")
for d in digits:
    print(f" {d:>7}", end="")
print()
print(f"  {'Legit':<8}", end="")
for d in digits:
    print(f" {zs_legit[d]:>7.2f}", end="")
print()
print(f"  {'Wash':<8}", end="")
for d in digits:
    print(f" {zs_wash[d]:>7.2f}", end="")
print()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, zsc, label, color in [
    (axes[0], zs_legit, "Legitimate wallets", CB["blue"]),
    (axes[1], zs_wash,  "Wash traders",       CB["red"]),
]:
    vals = [zsc[d] for d in digits]
    bars = ax.bar(digits, vals, color=color, alpha=0.85, width=0.6)
    ax.axhline(1.96, color=CB["orange"], linestyle="--", linewidth=1.2,
               label="z = 1.96 (p < 0.05)")
    ax.set_xlabel("Leading digit")
    ax.set_ylabel("Z-score")
    ax.set_title(f"Per-digit Z-scores — {label}")
    ax.set_xticks(digits)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()


---
## 5  Animated time-series: MAD as a wash-trade ring builds up

In the real pipeline, trades arrive continuously.  This section simulates a
**wash-trade ring gradually increasing its activity** — starting with a mix of
organic trades, then progressively replacing them with fixed-lot bot trades.

We compute the Benford MAD at each step and plot the trajectory, showing how
it crosses the 0.015 non-conformity threshold once the ring dominates the
volume.  (The plot is static for CI compatibility; a live Jupyter environment
renders this with `matplotlib` animation controls.)


In [ ]:
STEPS = 40
mad_trajectory = []

for step in range(STEPS):
    wash_fraction = step / (STEPS - 1)        # 0 → 1 linearly
    n_organic = int(N_TRADES * (1 - wash_fraction))
    n_bot     = N_TRADES - n_organic

    organic = pd.Series(10 ** RNG.uniform(0, 5, size=n_organic))
    bots    = pd.Series(RNG.choice([100.0, 1000.0, 5000.0], size=n_bot))
    combined = pd.concat([organic, bots], ignore_index=True)

    mad_trajectory.append(mad_score(combined))

fig, ax = plt.subplots(figsize=(9, 4))
steps = np.linspace(0, 100, STEPS)

ax.plot(steps, mad_trajectory, color=CB["blue"], linewidth=2, label="MAD score")
ax.axhline(MAD_NONCONFORMITY_THRESHOLD, color=CB["red"], linestyle="--", linewidth=1.5,
           label=f"Non-conformity threshold ({MAD_NONCONFORMITY_THRESHOLD})")

# Shade the region where MAD exceeds the threshold
mad_arr = np.array(mad_trajectory)
ax.fill_between(steps, mad_arr, MAD_NONCONFORMITY_THRESHOLD,
                where=(mad_arr >= MAD_NONCONFORMITY_THRESHOLD),
                alpha=0.15, color=CB["red"], label="Flagged region")

ax.set_xlabel("Wash-trade ring activity (% of total volume)")
ax.set_ylabel("Benford MAD score")
ax.set_title("MAD score as wash-trade ring builds up\n(0 % = all organic, 100 % = all bot trades)")
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

crossing = next((s for s, m in zip(steps, mad_trajectory) if m >= MAD_NONCONFORMITY_THRESHOLD), None)
if crossing is not None:
    print(f"MAD crosses the non-conformity threshold at ≈ {crossing:.0f} % wash activity.")


---
## 6  Rolling-window analysis

LedgerLens computes all three Benford metrics over **five standard time windows**:
`1 h`, `4 h`, `24 h`, `7 d`, `30 d`.  Shorter windows are more sensitive to
bursts; longer windows capture sustained patterns.

Here we read the **pre-computed** Benford features from the synthetic parquet
directly — the feature columns follow the naming convention
`benford_chi_square_{h}h`, `benford_mad_{h}h`, `benford_z_max_{h}h`.

We show how those feature values compare between the two groups across all five
windows using grouped box-plots.


In [ ]:
WINDOWS = [1, 4, 24, 168, 720]
WINDOW_LABELS = ["1 h", "4 h", "24 h", "7 d", "30 d"]

legit_df = df[df["label"] == 0]
wash_df  = df[df["label"] == 1]

# ── MAD across windows ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

positions_legit = np.arange(len(WINDOWS)) * 3.0
positions_wash  = positions_legit + 1.1

bp_legit = ax.boxplot(
    [legit_df[f"benford_mad_{h}h"].dropna() for h in WINDOWS],
    positions=positions_legit, widths=0.8, patch_artist=True,
    boxprops=dict(facecolor=CB["blue"], alpha=0.6),
    medianprops=dict(color=CB["black"], linewidth=2),
    whiskerprops=dict(color=CB["blue"]),
    capprops=dict(color=CB["blue"]),
    flierprops=dict(marker="o", markersize=3, color=CB["blue"], alpha=0.4),
)
bp_wash = ax.boxplot(
    [wash_df[f"benford_mad_{h}h"].dropna() for h in WINDOWS],
    positions=positions_wash, widths=0.8, patch_artist=True,
    boxprops=dict(facecolor=CB["red"], alpha=0.6),
    medianprops=dict(color=CB["black"], linewidth=2),
    whiskerprops=dict(color=CB["red"]),
    capprops=dict(color=CB["red"]),
    flierprops=dict(marker="o", markersize=3, color=CB["red"], alpha=0.4),
)

ax.axhline(MAD_NONCONFORMITY_THRESHOLD, color=CB["orange"], linestyle="--",
           linewidth=1.5, label=f"Threshold ({MAD_NONCONFORMITY_THRESHOLD})")
ax.set_xticks(positions_legit + 0.55)
ax.set_xticklabels(WINDOW_LABELS)
ax.set_xlabel("Rolling window")
ax.set_ylabel("Benford MAD score")
ax.set_title("Benford MAD — legitimate vs. wash-trade wallets across rolling windows")

legend_elements = [
    Patch(facecolor=CB["blue"],   alpha=0.6, label="Legitimate"),
    Patch(facecolor=CB["red"],    alpha=0.6, label="Wash traders"),
    plt.Line2D([0], [0], color=CB["orange"], linestyle="--", label="Threshold"),
]
ax.legend(handles=legend_elements, fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# ── Chi-square across windows ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

bp_legit2 = ax.boxplot(
    [legit_df[f"benford_chi_square_{h}h"].dropna() for h in WINDOWS],
    positions=positions_legit, widths=0.8, patch_artist=True,
    boxprops=dict(facecolor=CB["blue"], alpha=0.6),
    medianprops=dict(color=CB["black"], linewidth=2),
    whiskerprops=dict(color=CB["blue"]),
    capprops=dict(color=CB["blue"]),
    flierprops=dict(marker="o", markersize=3, color=CB["blue"], alpha=0.4),
)
bp_wash2 = ax.boxplot(
    [wash_df[f"benford_chi_square_{h}h"].dropna() for h in WINDOWS],
    positions=positions_wash, widths=0.8, patch_artist=True,
    boxprops=dict(facecolor=CB["red"], alpha=0.6),
    medianprops=dict(color=CB["black"], linewidth=2),
    whiskerprops=dict(color=CB["red"]),
    capprops=dict(color=CB["red"]),
    flierprops=dict(marker="o", markersize=3, color=CB["red"], alpha=0.4),
)

ax.axhline(15.51, color=CB["orange"], linestyle="--", linewidth=1.5,
           label="χ² critical value (df=8, p=0.05) ≈ 15.5")
ax.set_xticks(positions_legit + 0.55)
ax.set_xticklabels(WINDOW_LABELS)
ax.set_xlabel("Rolling window")
ax.set_ylabel("Chi-square statistic")
ax.set_title("Benford chi-square — legitimate vs. wash-trade wallets across rolling windows")
legend_elements2 = [
    Patch(facecolor=CB["blue"],   alpha=0.6, label="Legitimate"),
    Patch(facecolor=CB["red"],    alpha=0.6, label="Wash traders"),
    plt.Line2D([0], [0], color=CB["orange"], linestyle="--", label="χ² critical value"),
]
ax.legend(handles=legend_elements2, fontsize=10)
plt.tight_layout()
plt.show()


---
## 7  Single-wallet deep-dive

Let's pick one representative wallet from each group and print a full
`BenfordMetrics` summary using the engine's `compute_benford_metrics()`
function directly on reconstructed trade amounts.


In [ ]:
# Representative wallets
wallet_legit = "GSYNTH000000"   # first legitimate wallet (synthetic ID)
wallet_wash  = "GSYNTH000250"   # first wash-trade wallet

# Reconstruct amounts for a single wallet
rng2 = np.random.default_rng(7)
single_legit_amounts = pd.Series(10 ** rng2.uniform(0, 5, size=120))
single_wash_amounts  = pd.Series(rng2.choice([100.0, 500.0, 1000.0], size=120))

for wallet_label, amounts in [("Legitimate", single_legit_amounts),
                                ("Wash trader", single_wash_amounts)]:
    m = compute_benford_metrics(amounts)
    print(f"=== {wallet_label} ===")
    print(f"  Sample size  : {m.sample_size}")
    print(f"  Chi-square   : {m.chi_square:.3f}")
    print(f"  MAD          : {m.mad:.4f}  (non-conforming: {m.mad_nonconforming})")
    print(f"  Z-scores     : { {d: round(v, 2) for d, v in m.z_scores.items()} }")
    print()


---
## 8  What Benford's Law can (and cannot) detect

### Strengths
- **Algorithm-agnostic**: any bot that uses systematic amounts breaks the law,
  regardless of the specific strategy.
- **Explainable**: a per-digit Z-score tells an investigator exactly which digit
  is anomalous, making the signal auditable.
- **Fast**: all three metrics are O(n) in trade count.

### Limitations
- **High-frequency market makers** can legitimately produce non-Benford
  distributions when trading tight spreads in thin markets.
- **Small sample sizes** produce wide confidence intervals; LedgerLens requires
  a minimum of trades before flagging.
- **Adversarial conformance**: a sophisticated attacker who knows about Benford
  can craft amounts to conform — LedgerLens counters this with second-digit
  analysis and ensemble ML features.

This is why Benford metrics are combined with 30+ on-chain ML features
(counterparty concentration, round-trip frequency, wallet graph topology, etc.)
to produce the final **LedgerLens Risk Score**.

---

### Further reading

- Benford, F. (1938). *The law of anomalous numbers.*
  Proceedings of the American Philosophical Society, 78(4), 551–572.
- Nigrini, M. J. (2012). *Benford's Law: Applications for Forensic Accounting,
  Auditing, and Fraud Detection.* Wiley.
- [LedgerLens GitHub](https://github.com/Ledger-Lenz/Ledgerlens-data)
- [`detection/benford_engine.py`](../detection/benford_engine.py) — production source
